# CatBoost в рефакторинг-пайплайне

Ноутбук оставлен как специализированный этап для сильной глобальной модели. В рефакторинге он должен использовать общие сплиты и более осторожный набор признаков без `sales_sum`.


## Политика признаков

- `sales_sum` исключен из базового списка признаков как потенциальная утечка.
- Числовые признаки для финального submission должны существовать и в `train_processed`, и в `test_processed`.
- `transactions` и `dcoilwtico` можно использовать только после отдельной проверки доступности на тестовом горизонте.


In [9]:
# %pip install catboost
# %pip install optuna

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [optuna]2m4/5 [optuna]]]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
import optuna

import warnings
warnings.filterwarnings("ignore")

# Настройка pandas
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Логгер
import logging

logger = logging.getLogger("catboost_store_level")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger catboost_store_level initialized")


2025-11-21 17:25:43,808 - catboost_store_level - INFO - Logger catboost_store_level initialized


In [2]:
DATA_DIR = "../data"
EXP_INFO_DIR = "../data/experiment_info"

train_path = os.path.join(DATA_DIR, "train_processed.csv")
test_path  = os.path.join(DATA_DIR, "test_processed.csv")
splits_path = os.path.join(EXP_INFO_DIR, "splits.json")

logger.info(f"Loading train from {train_path}")
train_processed = pd.read_csv(train_path, parse_dates=["date"])

logger.info(f"Loading test from {test_path}")
test_processed = pd.read_csv(test_path, parse_dates=["date"])

logger.info(f"Loading splits from {splits_path}")
with open(splits_path, "r", encoding="utf-8") as f:
    splits_json = json.load(f)

len(train_processed), len(test_processed), len(splits_json)


2025-11-21 17:25:43,830 - catboost_store_level - INFO - Loading train from ../data/train_processed.csv
2025-11-21 17:26:00,570 - catboost_store_level - INFO - Loading test from ../data/test_processed.csv
2025-11-21 17:26:00,649 - catboost_store_level - INFO - Loading splits from ../data/experiment_info/splits.json


(3008016, 28512, 3)

In [8]:
# Восстановим список сплитов в удобном формате
splits = []
for s in splits_json:
    splits.append(
        {
            "name": s["name"],
            "train_idx": pd.Index(s["train_idx"]),
            "val_idx": pd.Index(s["val_idx"]),
            "train_end": pd.to_datetime(s["train_end"]),
            "val_start": pd.to_datetime(s["val_start"]),
            "val_end": pd.to_datetime(s["val_end"]),
        }
    )

# Быстрый просмотр
rows = []
for s in splits:
    tr = train_processed.loc[s["train_idx"]]
    va = train_processed.loc[s["val_idx"]]
    rows.append(
        {
            "name": s["name"],
            "train_start": tr["date"].min().date(),
            "train_end": tr["date"].max().date(),
            "val_start": va["date"].min().date(),
            "val_end": va["date"].max().date(),
            "train_n": len(tr),
            "val_n": len(va),
        }
    )

splits_overview = pd.DataFrame(rows)
splits_overview


,name,train_start,train_end,val_start,val_end,train_n,val_n
0,split_1,2013-01-01,2017-07-30,2017-07-31,2017-08-15,2979504,28512
1,split_2,2013-01-01,2017-06-30,2017-07-01,2017-07-16,2926044,28512
2,split_3,2013-01-01,2017-05-31,2017-06-01,2017-06-16,2872584,28512


In [9]:
# Считаем суммарные продажи по (store_nbr, family)
series_sales = (
    train_processed
    .groupby(["store_nbr", "family"])["sales"]
    .sum()
    .rename("sales_sum")
    .reset_index()
)

logger.info(f"Total number of (store, family) series: {len(series_sales)}")

# Общий оборот
total_sales = series_sales["sales_sum"].sum()

# Берём топ-20% по sales_sum
quantile_cut = series_sales["sales_sum"].quantile(0.8)
top_series = series_sales[series_sales["sales_sum"] >= quantile_cut].copy()

logger.info(
    f"Selected {len(top_series)} top series (>= 80th percentile by sales_sum)"
)

top_sales_share = top_series["sales_sum"].sum() / total_sales
logger.info(
    f"Top series cover {top_sales_share:.3%} of total sales"
)

top_series.head()


2025-11-18 07:01:16,207 - catboost_store_level - INFO - Total number of (store, family) series: 1782
2025-11-18 07:01:16,212 - catboost_store_level - INFO - Selected 357 top series (>= 80th percentile by sales_sum)
2025-11-18 07:01:16,213 - catboost_store_level - INFO - Top series cover 88.598% of total sales


,store_nbr,family,sales_sum
3,1,BEVERAGES,2.673769e+06
5,1,BREAD/BAKERY,5.699922e+05
7,1,CLEANING,1.078525e+06
8,1,DAIRY,1.054354e+06
12,1,GROCERY I,3.743823e+06


In [10]:
top_pairs = set(
    top_series[["store_nbr", "family"]].itertuples(index=False, name=None)
)

len(top_pairs)

def filter_to_top_pairs(df):
    """
    Возвращает подтаблицу df только по выбранным top_pairs (store_nbr, family).
    """
    mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()


## Временные и категориальные фичи для CatBoost

Для CatBoost будем использовать:

- **временные фичи**:
  - день недели, номер недели, месяц, день месяца;
- **календари/праздники**:
  - `is_holiday`, `is_payday`;
- **промо и поведение магазина**:
  - `onpromotion`, `transactions` (где доступно);
- **лаговые фичи по продажам**:
  - лаги `sales` на 1, 7, 14, 28 дней;
  - скользящее среднее по 7 и 28 дням;
- **категориальные фичи**:
  - `store_nbr`, `family`, `city`, `state`, `store_type`, `cluster`,
    `type`, `locale`, `agg_description`.

CatBoost сам умеет работать с категориальными признаками, так что будем
передавать их в виде `str`/`int` и указывать список `cat_features`.

In [3]:
def add_time_features(df):
    """
    Добавляет базовые временные признаки в df:
    - dayofweek, weekofyear, month, day, is_weekend.
    Ожидает колонку 'date' типа datetime64[ns].
    """
    df = df.copy()
    df["dow"] = df["date"].dt.weekday.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["is_weekend"] = df["dow"].isin([5, 6]).astype("int8")
    return df

In [4]:
def add_lag_features(df, lags, group_cols=["store_nbr", "family"], target_col="sales"):
    """
    Добавляет лаги target_col на заданные сдвиги в днях.
    """
    df = df.sort_values(["date"] + group_cols).copy()
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = (
            df.groupby(group_cols)[target_col]
            .shift(lag)
        )
    return df

def add_rolling_features(df, windows, group_cols=["store_nbr", "family"], target_col="sales"):
    df = df.sort_values(["date"] + group_cols).copy()
    g = df.groupby(group_cols)[target_col]
    for w in windows:
        df[f"{target_col}_rolling_mean_{w}"] = (
            g.shift(1).rolling(window=w, min_periods=1).mean()
        )
    return df

In [5]:
# Список категориальных признаков для CatBoost
CAT_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "store_type",
    "cluster",
    "type",
    "locale",
    "agg_description",
]

# Бинарные/индикаторные признаки
BIN_FEATURES = [
    "is_holiday",
    "is_payday",
    "transferred",
]

# Базовые безопасные численные признаки для общего и submission-режима
SAFE_NUM_FEATURES_BASE = [
    "onpromotion",
]

# Эти признаки можно возвращать только после отдельной проверки доступности
# на тестовом горизонте и отсутствия методологических проблем.
OPTIONAL_RESEARCH_NUM_FEATURES = [
    col for col in ["transactions", "dcoilwtico"]
    if col in train_processed.columns and col in test_processed.columns
]

NUM_FEATURES_BASE = SAFE_NUM_FEATURES_BASE + OPTIONAL_RESEARCH_NUM_FEATURES

# Временные фичи, которые добавим
TIME_FEATURES = ["dow", "weekofyear", "month", "day", "is_weekend"]


In [14]:
def make_train_val_for_split(
    split,
    full_df,
    top_pairs,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит X_train, y_train, X_val, y_val для заданного сплита.

    Параметры:
    - split: словарь из splits (name, train_idx, val_idx, ...)
    - full_df: train_processed
    - top_pairs: множество (store_nbr, family)
    - lags: tuple лагов для sales
    - roll_windows: окна для скользящего среднего по sales

    Возвращает:
    - X_train, y_train, X_val, y_val, cat_features
    """
    split_name = split["name"]
    logger.info(f"Preparing data for {split_name}")

    # Берём train/val по индексам
    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    # Фильтрация по top_pairs
    def _filter(df):
        mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
        return df[mask].copy()

    train_df = _filter(train_df)
    val_df   = _filter(val_df)

    logger.info(
        f"{split_name} | train rows after top_pairs filter: {len(train_df)}, "
        f"val rows: {len(val_df)}"
    )

    # Склеиваем train+val для корректного расчёта лагов (история перед валидацией)
    combined = pd.concat([train_df, val_df], axis=0).sort_values(["store_nbr", "family", "date"])

    # Добавляем временные фичи
    combined = add_time_features(combined)

    # Лаги по sales
    combined = add_lag_features(combined, lags=lags, group_cols=["store_nbr", "family"], target_col="sales")

    # Скользящие средние
    combined = add_rolling_features(combined, windows=roll_windows, group_cols=["store_nbr", "family"], target_col="sales")

    # После лагов/rolling первые дни ряда будут с NaN — отрежем их
    # Но делаем это аккуратно: только там, где NaN в хотя бы одном из lag/rolling.
    feature_cols_lag_roll = [col for col in combined.columns if "lag_" in col or "rolling_mean_" in col]
    combined = combined.dropna(subset=feature_cols_lag_roll)

    # Разделяем обратно на train/val по дате
    train_end = train_df["date"].max()
    val_start = val_df["date"].min()
    val_end   = val_df["date"].max()

    train_mask = (combined["date"] <= train_end)
    val_mask   = (combined["date"] >= val_start) & (combined["date"] <= val_end)

    train_fe = combined[train_mask].copy()
    val_fe   = combined[val_mask].copy()

    logger.info(
        f"{split_name} | after lag/rolling dropna: train={len(train_fe)}, val={len(val_fe)}"
    )

    # Целевая переменная
    y_train = train_fe["sales"].astype("float32")
    y_val   = val_fe["sales"].astype("float32")

    # Формируем список всех фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )

    # Оставляем только существующие в датафрейме
    feature_cols = [c for c in feature_cols if c in train_fe.columns]

    X_train = train_fe[feature_cols].copy()
    X_val   = val_fe[feature_cols].copy()

    # CatBoost требует индексы категориальных колонок
    cat_features = [i for i, c in enumerate(feature_cols) if c in CAT_FEATURES + BIN_FEATURES]

    logger.info(
        f"{split_name} | X_train shape={X_train.shape}, X_val shape={X_val.shape}, "
        f"n_cat_features={len(cat_features)}"
    )

    return X_train, y_train, X_val, y_val, feature_cols, cat_features


In [6]:
def rmsle_metric(y_true, y_pred):
    """
    RMSLE в исходном масштабе (после обратного expm1).
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

In [12]:
optuna_split = splits[0]
logger.info(f"Optuna will use split: {optuna_split['name']}")

def objective(trial):
    """
    Целевая функция для Optuna:
    - выбирает гиперпараметры CatBoost;
    - обучает модель на log1p(sales) на train части optuna_split;
    - возвращает RMSLE на val части optuna_split в исходном масштабе.
    """
    # Гиперпараметры для поиска
    param = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 64, 255),
    }

    # Данные для сплита
    X_train, y_train, X_val, y_val, feature_cols, cat_features = make_train_val_for_split(
        optuna_split,
        train_processed,
        top_pairs,
        lags=(1, 7, 14, 28),
        roll_windows=(7, 28),
    )

    # Лог-преобразование таргета
    y_train_log = np.log1p(y_train)
    y_val_log   = np.log1p(y_val)

    train_pool = Pool(
        data=X_train,
        label=y_train_log,
        cat_features=cat_features,
    )
    val_pool = Pool(
        data=X_val,
        label=y_val_log,
        cat_features=cat_features,
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=param["depth"],
        learning_rate=param["learning_rate"],
        l2_leaf_reg=param["l2_leaf_reg"],
        bagging_temperature=param["bagging_temperature"],
        border_count=param["border_count"],
        random_state=42,
        thread_count=4,
        task_type="CPU",
        verbose=False,
        od_type="Iter",
        od_wait=50,
        iterations=3000,
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        verbose=False,
    )

    # Предсказания в лог-пространстве → обратно в исходный масштаб
    y_pred_log = model.predict(val_pool)
    y_pred = np.expm1(y_pred_log)

    score = rmsle_metric(y_val, y_pred)

    logger.info(
        f"Trial {trial.number}: params={param}, RMSLE={score:.4f}"
    )

    return score


2025-11-16 20:10:14,119 - catboost_store_level - INFO - Optuna will use split: split_1


Стандартный и рабочий подход: обучать модель на log1p(sales) с loss_function="RMSE", а потом обратно приводить прогноз через expm1, а для отчёта считать RMSLE уже руками — это по сути то же самое, что минимизировать RMSLE, но в рамках поддерживаемой CatBoost формулировки.

In [ ]:
import multiprocessing

n_cores = multiprocessing.cpu_count()
logger.info(f"CPU cores available: {n_cores}")

study = optuna.create_study(
    direction="minimize",
    study_name="catboost_rmsle_log1p",
)

N_TRIALS = 20  # можно увеличить/уменьшить по ситуации

logger.info(f"Starting Optuna optimization with {N_TRIALS} trials...")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=2,  # параллель по всем ядрам
    show_progress_bar=True,
)

logger.info(f"Best RMSLE: {study.best_value:.4f}")
logger.info(f"Best params: {study.best_params}")

2025-11-16 20:10:14,141 - catboost_store_level - INFO - CPU cores available: 8
[I 2025-11-16 20:10:14,142] A new study created in memory with name: catboost_rmsle_log1p
2025-11-16 20:10:14,144 - catboost_store_level - INFO - Starting Optuna optimization with 20 trials...
  0%|          | 0/20 [00:00<?, ?it/s]2025-11-16 20:10:14,149 - catboost_store_level - INFO - Preparing data for split_1
2025-11-16 20:10:14,152 - catboost_store_level - INFO - Preparing data for split_1
2025-11-16 20:10:45,407 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 20:10:45,412 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 20:10:48,743 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 20:10:48,847 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 20:10:48,946 - catboost_store_level

[I 2025-11-16 20:17:07,583] Trial 1 finished with value: 0.14432644698886268 and parameters: {'depth': 8, 'learning_rate': 0.21818765331024817, 'l2_leaf_reg': 0.0010645567731311265, 'bagging_temperature': 2.9526580952292814, 'border_count': 73}. Best is trial 1 with value: 0.14432644698886268.


2025-11-16 20:17:25,307 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 20:17:27,759 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 20:17:27,877 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12
2025-11-16 20:44:22,089 - catboost_store_level - INFO - Trial 2: params={'depth': 9, 'learning_rate': 0.12201259966751375, 'l2_leaf_reg': 0.4266880256915675, 'bagging_temperature': 0.762075344882544, 'border_count': 78}, RMSLE=0.1344
Best trial: 2. Best value: 0.134411:  10%|█         | 2/20 [34:07<5:39:31, 1131.73s/it]2025-11-16 20:44:22,117 - catboost_store_level - INFO - Preparing data for split_1


[I 2025-11-16 20:44:22,107] Trial 2 finished with value: 0.13441120300672585 and parameters: {'depth': 9, 'learning_rate': 0.12201259966751375, 'l2_leaf_reg': 0.4266880256915675, 'bagging_temperature': 0.762075344882544, 'border_count': 78}. Best is trial 2 with value: 0.13441120300672585.


2025-11-16 20:44:38,047 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 20:44:40,668 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 20:44:40,774 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12
2025-11-16 20:45:06,585 - catboost_store_level - INFO - Trial 0: params={'depth': 9, 'learning_rate': 0.01241189873293173, 'l2_leaf_reg': 0.16328470906703352, 'bagging_temperature': 2.8658252350870077, 'border_count': 75}, RMSLE=0.1480
Best trial: 2. Best value: 0.134411:  15%|█▌        | 3/20 [34:52<2:59:59, 635.28s/it] 2025-11-16 20:45:06,617 - catboost_store_level - INFO - Preparing data for split_1


...

2025-11-16 21:08:19,868 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 21:08:22,414 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 21:08:22,537 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12
2025-11-16 21:17:31,097 - catboost_store_level - INFO - Trial 7: params={'depth': 8, 'learning_rate': 0.1776098534593899, 'l2_leaf_reg': 0.00844822554503904, 'bagging_temperature': 3.6743270992424426, 'border_count': 92}, RMSLE=0.1383
Best trial: 2. Best value: 0.134411:  40%|████      | 8/20 [1:07:16<1:23:50, 419.19s/it]2025-11-16 21:17:31,120 - catboost_store_level - INFO - Preparing data for split_1


[I 2025-11-16 21:17:31,110] Trial 7 finished with value: 0.13834962257894673 and parameters: {'depth': 8, 'learning_rate': 0.1776098534593899, 'l2_leaf_reg': 0.00844822554503904, 'bagging_temperature': 3.6743270992424426, 'border_count': 92}. Best is trial 2 with value: 0.13441120300672585.


2025-11-16 21:17:56,733 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 21:18:00,175 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 21:18:00,299 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12
2025-11-16 21:24:28,129 - catboost_store_level - INFO - Trial 9: params={'depth': 10, 'learning_rate': 0.2041499419877062, 'l2_leaf_reg': 0.15806435439740055, 'bagging_temperature': 0.9213039069973472, 'border_count': 235}, RMSLE=0.1413
Best trial: 2. Best value: 0.134411:  45%|████▌     | 9/20 [1:14:14<1:16:43, 418.52s/it]2025-11-16 21:24:28,156 - catboost_store_level - INFO - Preparing data for split_1


[I 2025-11-16 21:24:28,143] Trial 9 finished with value: 0.14129326678450393 and parameters: {'depth': 10, 'learning_rate': 0.2041499419877062, 'l2_leaf_reg': 0.15806435439740055, 'bagging_temperature': 0.9213039069973472, 'border_count': 235}. Best is trial 2 with value: 0.13441120300672585.


2025-11-16 21:24:43,946 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-16 21:24:46,480 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-16 21:24:46,585 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12


In [ ]:
best_params = study.best_params
best_params["n_trials"] = N_TRIALS
best_params["best_value"] = study.best_value

exp_info_dir = Path("../data/experiment_info")
exp_info_dir.mkdir(parents=True, exist_ok=True)
params_path = exp_info_dir / "catboost_best_params.json"

with open(params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

logger.info(f"Best CatBoost params saved to {params_path}")
best_params

# Обучаем CatBoost на всех сплитах и считаем метрики

1. Загрузим json с лучшими гиперпараметрами, поскольку считали в облаке и после подбора параметров kermel остановился
2. Возьмем функцию из прошлого ноутбука для подсчета метрик
3. Обучим на всех сплитах и посчитаем метрики
4. Сделаем финальный submit в соревнование

In [8]:
with open("../data/experiment_info/catboost_best_params.json", "r", encoding="utf-8") as f:
    best_params = json.load(f)

In [17]:
def calculate_metrics_all(y_true, y_pred):
    """
    Возвращает dict с RMSLE, RMSE, MAE, WAPE.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Обрезаем прогнозы по нулю
    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE
    denom = np.sum(np.abs(y_true))
    wape = np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }


In [18]:
def fit_catboost_on_split(
    split,
    full_df,
    top_pairs,
    best_params,
    thread_count=4,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Обучает CatBoost на train-части сплита.
    Ничего не знает про валидацию и метрики — только fit.

    Возвращает:
    - model: обученный CatBoostRegressor
    - feature_cols: список фичей в том порядке, как их ждёт модель
    - cat_features_idx: индексы категориальных фичей в feature_cols
    """
    split_name = split["name"]
    logger.info(f"[{split_name}] Fitting CatBoost (train only)")

    # Берём старую утилиту, но будем использовать только train-часть
    X_train, y_train, X_val_dummy, y_val_dummy, feature_cols, cat_features_idx = make_train_val_for_split(
        split,
        full_df,
        top_pairs,
        lags=lags,
        roll_windows=roll_windows,
    )

    # Лог-преобразование таргета
    y_train_log = np.log1p(y_train)

    train_pool = Pool(
        data=X_train,
        label=y_train_log,
        cat_features=cat_features_idx,
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=best_params["depth"],
        learning_rate=best_params["learning_rate"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        bagging_temperature=best_params["bagging_temperature"],
        border_count=best_params["border_count"],
        random_state=42,
        thread_count=thread_count,
        task_type="CPU",
        verbose=False,
        od_type="Iter",
        od_wait=50,
        iterations=2000,  # можно потом уменьшить
    )

    start_time = datetime.now()
    logger.info(f"[{split_name}] CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    model.fit(
        train_pool,
        use_best_model=False,  # в этой функции только train, без val
        verbose=False,
    )

    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    logger.info(f"[{split_name}] CatBoost fit finished in {elapsed:.1f} seconds")

    return model, feature_cols, cat_features_idx

In [19]:
def prepare_history_and_future(split, full_df, top_pairs):
    """
    Готовит:
    - history_df: train-часть сплита (только top_pairs), отсортированную по (store_nbr, family, date);
    - future_df: val-часть сплита (только top_pairs), также отсортированную;
    - y_true: вектор истинных продаж для future_df (в том же порядке).
    """
    split_name = split["name"]

    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    def _filter(df):
        mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
        return df[mask].copy()

    train_df = _filter(train_df)
    val_df   = _filter(val_df)

    logger.info(
        f"[{split_name}] history rows (train, top_pairs)={len(train_df)}, "
        f"future rows (val, top_pairs)={len(val_df)}"
    )

    history_df = train_df.sort_values(["store_nbr", "family", "date"]).copy()
    future_df  = val_df.sort_values(["store_nbr", "family", "date"]).copy()

    y_true = future_df["sales"].astype("float32").values

    # В future_df 'sales' будем перезаписывать предсказаниями, поэтому сохраним y_true отдельно.
    future_df = future_df.drop(columns=["sales"])

    return history_df, future_df, y_true


In [20]:
def recursive_forecast_catboost_on_split(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Делает рекурсивный прогноз CatBoost на 16-дневном val-окне сплита.

    Шаги:
    - берём history_df (train) и future_df (val) из prepare_history_and_future;
    - для каждого дня val-окна:
        * добавляем его «каркас» (без sales) к истории;
        * пересчитываем временные, лаговые и rolling-фичи;
        * предсказываем sales на этот день;
        * записываем предсказания в историю, чтобы они участвовали в лаговых фичах следующего дня.
    Возвращает:
    - val_pred_df: DataFrame с колонками [date, store_nbr, family, y_true, y_pred]
    """
    split_name = split["name"]
    logger.info(f"[{split_name}] Recursive CatBoost forecast on validation window")

    history_df, future_df, y_true = prepare_history_and_future(split, full_df, top_pairs)

    # Будем накапливать историю с предсказаниями
    hist = history_df.copy()

    # Список предсказаний по дням
    preds_frames = []

    # Уникальные даты валидации по возрастанию
    val_dates = sorted(future_df["date"].unique())

    for cur_date in val_dates:
        logger.info(f"[{split_name}] Forecasting date {cur_date}")

        # Строки для текущего дня (exogenous features, без sales)
        day_future = future_df[future_df["date"] == cur_date].copy()

        # Временно добавляем пустой sales (NaN), чтобы лаги считались от hist
        day_future["sales"] = np.nan

        # Склеиваем историю + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи: время, лаги, rolling
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только текущий день уже с фичами
        feature_cols_lag_roll = [c for c in combined.columns if "lag_" in c or "rolling_mean_" in c]
        cur_df = combined[combined["date"] == cur_date].copy()

        # Отбрасываем строки, где нет необходимых лагов/rolling
        cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        # Если вдруг всё отвалилось (нехватка истории) — пропускаем с предупреждением
        if cur_df.empty:
            logger.warning(f"[{split_name}] No rows to predict for {cur_date}, skipping")
            continue

        # Готовим X для модели в том же порядке feature_cols
        X_cur = cur_df[feature_cols].copy()

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → назад в исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)

        cur_df["y_pred"] = y_pred

        # Истинные продажи для этого дня берём из full_df
        true_day = full_df[
            (full_df["date"] == cur_date)
            & (full_df[["store_nbr", "family"]]
               .apply(tuple, axis=1)
               .isin(top_pairs))
        ][["store_nbr", "family", "sales"]].rename(columns={"sales": "y_true"})

        cur_df = cur_df.merge(
            true_day,
            on=["store_nbr", "family"],
            how="left",
        )

        preds_frames.append(
            cur_df[["date", "store_nbr", "family", "y_true", "y_pred"]]
        )

        # Обновляем историю: для следующего дня 'sales' должны быть равны предсказанным y_pred
        # Берём ровно те строки, которые мы сейчас спрогнозировали (по ключам)
        hist_update = cur_df[["date", "store_nbr", "family"]].copy()
        hist_update["sales"] = y_pred

        # Также нужны остальные признаки (onpromotion, city, state, ...),
        # поэтому мёрджим их из future_df
        hist_update = hist_update.merge(
            future_df[future_df["date"] == cur_date].drop(columns=[]),
            on=["date", "store_nbr", "family"],
            how="left",
        )

        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)

    # Собираем всё вместе
    val_pred_df = pd.concat(preds_frames, axis=0, ignore_index=True)

    return val_pred_df


In [21]:
def evaluate_catboost_on_split_recursive(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
):
    """
    Обёртка: рекурсивный прогноз + расчёт метрик на сплите.
    Ответственность: только оценка качества.
    """
    split_name = split["name"]

    val_pred_df = recursive_forecast_catboost_on_split(
        split=split,
        full_df=full_df,
        top_pairs=top_pairs,
        model=model,
        feature_cols=feature_cols,
        cat_features_idx=cat_features_idx,
    )

    y_true = val_pred_df["y_true"].values
    y_pred = val_pred_df["y_pred"].values

    metrics = calculate_metrics_all(y_true, y_pred)
    metrics["split"] = split_name
    metrics["model"] = "catboost_store_family_recursive"

    logger.info(
        f"[{split_name}] CatBoost (recursive) metrics: "
        f"RMSLE={metrics['RMSLE']:.4f}, "
        f"RMSE={metrics['RMSE']:.2f}, "
        f"MAE={metrics['MAE']:.2f}, "
        f"WAPE={metrics['WAPE']:.4f}"
    )

    return metrics, val_pred_df


In [ ]:
catboost_metrics_by_split = {}
catboost_preds_by_split = {}

THREAD_COUNT = -1

for s in splits:
    split_name = s["name"]
    logger.info(f"=== {split_name} | CatBoost recursive 16-day forecast ===")

    # 1. Обучение
    model, feature_cols, cat_features_idx = fit_catboost_on_split(
        split=s,
        full_df=train_processed,
        top_pairs=top_pairs,
        best_params=best_params,
        thread_count=THREAD_COUNT,
    )

    # 2. Рекурсивный прогноз + метрики
    metrics, val_pred_df = evaluate_catboost_on_split_recursive(
        split=s,
        full_df=train_processed,
        top_pairs=top_pairs,
        model=model,
        feature_cols=feature_cols,
        cat_features_idx=cat_features_idx,
    )

    catboost_metrics_by_split[split_name] = metrics
    catboost_preds_by_split[split_name] = val_pred_df

2025-11-18 05:11:06,564 - catboost_store_level - INFO - === split_1 | CatBoost recursive 16-day forecast ===
2025-11-18 05:11:06,565 - catboost_store_level - INFO - [split_1] Fitting CatBoost (train only)
2025-11-18 05:11:06,566 - catboost_store_level - INFO - Preparing data for split_1
2025-11-18 05:11:20,140 - catboost_store_level - INFO - split_1 | train rows after top_pairs filter: 596904, val rows: 5712
2025-11-18 05:11:21,936 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2025-11-18 05:11:22,049 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 27), X_val shape=(5712, 27), n_cat_features=12
2025-11-18 05:11:25,911 - catboost_store_level - INFO - [split_1] CatBoost fit started at 2025-11-18 05:11:25
2025-11-18 05:16:30,669 - catboost_store_level - INFO - [split_1] CatBoost fit finished in 304.8 seconds
2025-11-18 05:16:30,699 - catboost_store_level - INFO - [split_1] Recursive CatBoost forecast on validation window
2025-11

In [ ]:
# Преобразуем словарь catboost_metrics_by_split в DataFrame
rows = []
for split_name, m in catboost_metrics_by_split.items():
    rows.append(
        {
            "split": split_name,
            "model": m.get("model", "catboost_store_family_recursive"),
            "RMSLE": m["RMSLE"],
            "RMSE": m["RMSE"],
            "MAE": m["MAE"],
            "WAPE": m["WAPE"],
        }
    )

catboost_metrics_df = pd.DataFrame(rows).set_index("split")

# Добавим строку со средними по всем сплитам
mean_row = catboost_metrics_df.mean(axis=0).to_dict()
mean_row = pd.DataFrame(mean_row, index=["mean"])
catboost_metrics_df = pd.concat([catboost_metrics_df, mean_row])

# Путь для сохранения
metrics_path = "../data/experiment_info/catboost_store_metrics_by_split_recursive.csv"
catboost_metrics_df.to_csv(metrics_path, index=True)

logger.info(f"CatBoost recursive-forecast metrics saved to {metricss_path}")
metrics_path


## Обучение модели для финального submit

In [9]:
def fill_na_in_cat(df, feature_cols, cat_features_idx):
    """
    Заполняет NaN в категориальных колонках специальным токеном "__MISSING__".
    Это нужно, т.к. CatBoost не принимает NaN в cat_features. [web:472][web:475]
    """
    df = df.copy()
    cat_cols = [feature_cols[i] for i in cat_features_idx]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("__MISSING__")
    return df

In [10]:
def make_full_train_dataset(
    train_df,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит полный датасет для обучения финальной CatBoost-модели
    и возвращает X_full, y_full, feature_cols, cat_features_idx, dates_aligned.
    """
    # Сортируем, чтобы лаги считались корректно
    df = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Временные фичи
    df = add_time_features(df)

    # Лаги и rolling по sales
    df = add_lag_features(
        df,
        lags=lags,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )
    df = add_rolling_features(
        df,
        windows=roll_windows,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )

    # Все lag/rolling-фичи
    feature_cols_lag_roll = [
        c for c in df.columns
        if "lag_" in c or "rolling_mean_" in c
    ]

    # Отбрасываем строки без всех лагов/rolling
    df = df.dropna(subset=feature_cols_lag_roll)

    # Цель
    y_full = df["sales"].astype("float32")

    # Полный список фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )
    feature_cols = [c for c in feature_cols if c in df.columns]

    X_full = df[feature_cols].copy()

    cat_features_idx = [
        i for i, c in enumerate(feature_cols)
        if c in (CAT_FEATURES + BIN_FEATURES)
    ]

    # Даты в том же порядке, что X_full / y_full
    dates_aligned = df["date"].values

    logger.info(
        f"[FULL] Dataset: X_full.shape={X_full.shape}, y_full.len={len(y_full)}, "
        f"n_cat_features={len(cat_features_idx)}"
    )

    return X_full, y_full, feature_cols, cat_features_idx, dates_aligned


In [11]:
# Строим полный датасет
X_full, y_full, feature_cols_full, cat_features_idx_full, dates_full = make_full_train_dataset(
    train_processed,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)

# Последняя дата (валидация на один день)
last_date = dates_full.max()
logger.info(f"[FULL] Validation date = {pd.to_datetime(last_date).date()}")

val_mask = dates_full == last_date
train_mask = dates_full < last_date

X_train_full = X_full[train_mask]
y_train_full = y_full[train_mask]

X_val_full = X_full[val_mask]
y_val_full = y_full[val_mask]

logger.info(
    f"[FULL] Train size = {X_train_full.shape}, val size (1 day) = {X_val_full.shape}"
)


2025-11-21 17:27:04,372 - catboost_store_level - INFO - [FULL] Dataset: X_full.shape=(2958120, 27), y_full.len=2958120, n_cat_features=12
2025-11-21 17:27:04,440 - catboost_store_level - INFO - [FULL] Validation date = 2017-08-15
2025-11-21 17:27:04,803 - catboost_store_level - INFO - [FULL] Train size = (2956338, 27), val size (1 day) = (1782, 27)


In [12]:
THREAD_COUNT = -1  

# Обрабатываем NaN в категориальных фичах
X_train_full_filled = fill_na_in_cat(X_train_full, feature_cols_full, cat_features_idx_full)
X_val_full_filled   = fill_na_in_cat(X_val_full, feature_cols_full, cat_features_idx_full)

y_train_log = np.log1p(y_train_full)
y_val_log   = np.log1p(y_val_full)

train_pool_full = Pool(
    data=X_train_full_filled,
    label=y_train_log,
    cat_features=cat_features_idx_full,
)
val_pool_full = Pool(
    data=X_val_full_filled,
    label=y_val_log,
    cat_features=cat_features_idx_full,
)

final_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=best_params["depth"],
    learning_rate=best_params["learning_rate"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    border_count=best_params["border_count"],
    random_state=42,
    thread_count=THREAD_COUNT,
    task_type="CPU",
    verbose=False,
    od_type="Iter",
    od_wait=50,
    iterations=2000,
)

start_time = datetime.now()
logger.info(f"[FULL] Final CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

final_model.fit(
    train_pool_full,
    eval_set=val_pool_full,
    use_best_model=True,
    verbose=False,
)

end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds()
logger.info(f"[FULL] Final CatBoost fit finished in {elapsed:.1f} seconds")


2025-11-21 17:27:24,713 - catboost_store_level - INFO - [FULL] Final CatBoost fit started at 2025-11-21 17:27:24
2025-11-21 17:41:07,713 - catboost_store_level - INFO - [FULL] Final CatBoost fit finished in 823.0 seconds


In [28]:
def recursive_forecast_catboost_on_test(
    model,
    train_df,
    test_df,
    feature_cols,
    cat_features_idx,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Рекурсивный 16-дневный прогноз на тесте:
    - история = весь train (с реальными sales, включая последний день);
    - для каждого дня теста:
        * добавляем его каркас (без sales) к истории,
        * пересчитываем фичи (time, lag, rolling),
        * предсказываем sales,
        * подставляем предсказания в историю, чтобы они участвовали в лагах
          на следующих днях;
    - в конце мержим прогнозы с test_df по (store_nbr, family, date),
      чтобы вернуть id и получить submission формата id,sales. [web:483][web:387]
    """
    # История: весь train
    hist = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Тест: каркас (есть id, store_nbr, family, date и все exogenous-признаки)
    future = test_df.sort_values(["date", "store_nbr", "family"]).copy()

    test_dates = sorted(future["date"].unique())
    all_preds = []

    for cur_date in test_dates:
        cur_ts = pd.to_datetime(cur_date)
        logger.info(f"[TEST] Forecasting date {cur_ts.date()}")

        # Строки теста для текущей даты
        day_future = future[future["date"] == cur_date].copy()

        # Временный столбец sales=NaN, чтобы лаги считались только из hist
        day_future["sales"] = np.nan

        # История + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи времени + lag/rolling по sales
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только строки текущего дня уже с фичами
        cur_df = combined[combined["date"] == cur_date].copy()

        # ВАЖНО: на тесте НЕ делаем dropna по lag/rolling — CatBoost умеет с ними работать
        # cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        if cur_df.empty:
            logger.warning(f"[TEST] No rows to predict for {cur_ts.date()}, skipping")
            continue

        # X в том же порядке фичей, что при обучении
        X_cur = cur_df[feature_cols].copy()
        X_cur = fill_na_in_cat(X_cur, feature_cols, cat_features_idx)

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)
        y_pred = np.clip(y_pred, 0, None)

        # Сохраняем прогнозы вместе с ключами
        cur_df = cur_df[["store_nbr", "family", "date"]].copy()
        cur_df["sales"] = y_pred
        all_preds.append(cur_df)

        # Обновляем историю: на cur_date sales = предсказания (для будущих лагов)
        hist_update = day_future.copy()
        hist_update["sales"] = y_pred
        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)

    # Собираем все прогнозы по ключам
    preds_df = pd.concat(all_preds, axis=0, ignore_index=True)

    # Мерджим с test_df, чтобы вернуть id для сабмита
    submission_df = (
        test_df[["id", "store_nbr", "family", "date"]]
        .merge(
            preds_df,
            on=["store_nbr", "family", "date"],
            how="left",
        )
    )

    # На всякий случай проверим покрытие id
    missing_ids = set(test_df["id"]) - set(submission_df["id"])
    if missing_ids:
        logger.warning(f"[TEST] Missing predictions for {len(missing_ids)} ids")

    # Финальный формат для Kaggle
    submission_df = submission_df[["id", "sales"]]

    return submission_df


In [29]:
submission_df = recursive_forecast_catboost_on_test(
    model=final_model,
    train_df=train_processed,
    test_df=test_processed,
    feature_cols=feature_cols_full,
    cat_features_idx=cat_features_idx_full,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)

submission = submission_df.sort_values("id")[["id", "sales"]]
submission.to_csv("submission_catboost_recursive_final.csv", index=False)


2025-11-21 18:08:43,422 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-16
2025-11-21 18:08:50,516 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-17
2025-11-21 18:08:58,202 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-18
2025-11-21 18:09:05,412 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-19
2025-11-21 18:09:12,592 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-20
2025-11-21 18:09:19,776 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-21
2025-11-21 18:09:26,963 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-22
2025-11-21 18:09:34,102 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-23
2025-11-21 18:09:41,192 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-24
2025-11-21 18:09:48,303 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-25
2025-11-21 18:09:55,444 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-26